In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report


sns.set(style="whitegrid")

In [ ]:

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print("{train_df.shape}")
print("{test_df.shape}")

Train shape: (630000, 15)
Test shape: (270000, 14)


In [ ]:
train_df.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [ ]:

X = train_df.drop(['id', 'Heart Disease'], axis=1)
y = train_df['Heart Disease']

X_test_final = test_df.drop(['id'], axis=1)

categorical_cols = [col for col in X.columns if X[col].dtype == 'object']
numerical_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

Categorical columns: []
Numerical columns: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Target classes: {le.classes_}")#0 and 1

Target classes: ['Absence' 'Presence']


In [6]:
# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

## Hyperparameter Tuning

We use GridSearchCV to find optimal parameters instead of guessing.

In [7]:
# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Define the model with 'hist' tree method for much faster training
xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# Create pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', xgb_model)])

# Define a SMALLER parameter grid for faster tuning
param_grid = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [4, 6],
    'model__subsample': [0.8],
}

# Perform Grid Search with Stratified Cross-Validation (reduced to 3 folds)
# Added n_jobs=-1 to use all cores
grid_search = GridSearchCV(pipeline, param_grid, cv=StratifiedKFold(n_splits=3), scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Get the best model
best_model = grid_search.best_estimator_

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/Users/sithijaseneviratne/Documents/GitHub/Machine-Learning-Training/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [11:13:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/sithijaseneviratne/Documents/GitHub/Machine-Learning-Training/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [11:13:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/sithijaseneviratne/Documents/GitHub/Machine-Learning-Training/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [11:13:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/sithijaseneviratne/Documents/GitHub/Machine-Learning-T

Best parameters found: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best cross-validation accuracy: 0.8880


In [11]:

y_pred = best_model.predict(X_val)


print(accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

0.8893492063492063
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.87      0.88     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.89      0.89    126000
weighted avg       0.89      0.89      0.89    126000



In [ ]:
if 'id' in test_df.columns:
    test_ids = test_df['id']
    

test_preds = best_model.predict(X_test_final)

submission = pd.DataFrame({'id': test_ids, 'Heart Disease': test_preds})
submission.head()

,id,Heart Disease
0,630000,1
1,630001,0
2,630002,1
3,630003,0
4,630004,0


In [ ]:
# Save submission file
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")